In [1]:
# AI-Enhanced Dataset
import pandas as pd
import numpy as np

# Load AI-enhanced product dataset
product_data = pd.read_csv("ai_enhanced_product_data.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", product_data.shape)

print("\nColumns:")
print(product_data.columns.tolist())

product_data.head()

Dataset loaded successfully!
Dataset shape: (213, 10)

Columns:
['product_id', 'brand', 'product_type', 'price_range', 'item_price', 'product_rating', 'average_sentiment_score', 'positive_review_percentage', 'total_reviews', 'cluster']


,product_id,brand,product_type,price_range,item_price,product_rating,average_sentiment_score,positive_review_percentage,total_reviews,cluster
0,P001247,Nimbus,Jeans,Mid Range ($50-$200),67.54,3.58,0.666669,83.333333,6,4
1,P001002,Harbor,Skirt,Premium ($200-$500),209.10,3.68,0.776493,88.888889,9,1
2,P001301,Astra,Dress,Mid Range ($50-$200),114.51,2.87,0.999847,100.000000,6,2
3,P000162,Orion,Jacket,Mid Range ($50-$200),151.39,3.99,0.498688,75.000000,8,0
4,P000022,Nimbus,T-Shirt,Budget ($0-$50),37.65,4.54,0.665417,83.333333,6,1


In [2]:
# Check Missing Values
print("Missing values:")

print(
    product_data.isnull().sum()
)

Missing values:
product_id                    0
brand                         0
product_type                  0
price_range                   0
item_price                    0
product_rating                0
average_sentiment_score       0
positive_review_percentage    0
total_reviews                 0
cluster                       0
dtype: int64


### Create Text Documents for RAG

In [3]:
def create_product_document(row):

    return f"""
Product ID: {row['product_id']}

Brand: {row['brand']}

Product Type: {row['product_type']}

Price Range: {row['price_range']}

Price: ${row['item_price']}

Product Rating: {row['product_rating']}

Average Customer Sentiment Score:
{row['average_sentiment_score']}

Positive Review Percentage:
{row['positive_review_percentage']}%

Total Customer Reviews:
{row['total_reviews']}

Product Cluster:
{row['cluster']}
"""

In [4]:
product_data["rag_document"] = product_data.apply(
    create_product_document,
    axis=1
)

print("RAG documents created successfully!")

print("\nExample product document:\n")

print(
    product_data["rag_document"].iloc[0]
)

RAG documents created successfully!

Example product document:


Product ID: P001247

Brand: Nimbus

Product Type: Jeans

Price Range: Mid Range ($50-$200)

Price: $67.54

Product Rating: 3.58

Average Customer Sentiment Score:
0.6666691104571024

Positive Review Percentage:
83.33333333333334%

Total Customer Reviews:
6

Product Cluster:
4



In [5]:
%pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 8.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [sentence-transformers]ence-transformers]
Note: you may need to restart the kernel to use updated packages.


### Import the Required Libraries

In [6]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


### Load the Embedding Model

In [7]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


### Create Embeddings for Your Product Documents

In [8]:
product_embeddings = embedding_model.encode(
    product_data["rag_document"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embeddings created successfully!")

print("Embedding shape:")
print(product_embeddings.shape)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings created successfully!
Embedding shape:
(213, 384)


### Create the Vector Database

In [10]:
# create a FAISS vector database using your product embeddings
embedding_dimension = product_embeddings.shape[1]

vector_index = faiss.IndexFlatL2(
    embedding_dimension
)

vector_index.add(
    product_embeddings.astype("float32")
)

print("Vector database created successfully!")

print(
    "Total products stored:",
    vector_index.ntotal
)

Vector database created successfully!
Total products stored: 213


### Test the Vector Database

In [11]:
query = "Recommend affordable jeans with good customer reviews"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
)

distances, indices = vector_index.search(
    query_embedding.astype("float32"),
    k=5
)

print("Top 5 similar products:")

product_data.iloc[
    indices[0]
][
    [
        "product_id",
        "brand",
        "product_type",
        "price_range",
        "item_price",
        "product_rating",
        "positive_review_percentage"
    ]
]

Top 5 similar products:


,product_id,brand,product_type,price_range,item_price,product_rating,positive_review_percentage
36,P001733,Zenith,Jeans,Mid Range ($50-$200),166.64,3.39,75.000000
148,P001046,Zenith,Jeans,Mid Range ($50-$200),157.61,3.31,77.777778
170,P000539,Zenith,Jeans,Budget ($0-$50),45.11,3.38,50.000000
168,P000700,GreenLeaf,Jeans,Mid Range ($50-$200),186.13,2.87,77.777778
52,P001619,Everest,Jeans,Budget ($0-$50),15.52,4.20,83.333333


### Create a Function to Retrieve Relevant Products

In [12]:
def retrieve_products(query, k=5):

    # Convert user query into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    # Search the FAISS vector database
    distances, indices = vector_index.search(
        query_embedding.astype("float32"),
        k=k
    )

    # Get the matching products
    retrieved_products = product_data.iloc[
        indices[0]
    ].copy()

    return retrieved_products

In [13]:
query = "Recommend affordable jeans with good customer reviews"

results = retrieve_products(
    query,
    k=5
)

results[
    [
        "product_id",
        "brand",
        "product_type",
        "price_range",
        "item_price",
        "product_rating",
        "positive_review_percentage"
    ]
]

,product_id,brand,product_type,price_range,item_price,product_rating,positive_review_percentage
36,P001733,Zenith,Jeans,Mid Range ($50-$200),166.64,3.39,75.000000
148,P001046,Zenith,Jeans,Mid Range ($50-$200),157.61,3.31,77.777778
170,P000539,Zenith,Jeans,Budget ($0-$50),45.11,3.38,50.000000
168,P000700,GreenLeaf,Jeans,Mid Range ($50-$200),186.13,2.87,77.777778
52,P001619,Everest,Jeans,Budget ($0-$50),15.52,4.20,83.333333


### Prepare the Retrieved Products as Context for the LLM

In [14]:
def create_context(retrieved_products):

    context = ""

    for _, row in retrieved_products.iterrows():

        context += row["rag_document"]
        context += "\n-------------------\n"

    return context

In [15]:
query = "Recommend affordable jeans with good customer reviews"

results = retrieve_products(
    query,
    k=5
)

context = create_context(results)

print(context)


Product ID: P001733

Brand: Zenith

Product Type: Jeans

Price Range: Mid Range ($50-$200)

Price: $166.64

Product Rating: 3.39

Average Customer Sentiment Score:
0.4989480450749397

Positive Review Percentage:
75.0%

Total Customer Reviews:
8

Product Cluster:
4

-------------------

Product ID: P001046

Brand: Zenith

Product Type: Jeans

Price Range: Mid Range ($50-$200)

Price: $157.61

Product Rating: 3.31

Average Customer Sentiment Score:
0.5550151003731622

Positive Review Percentage:
77.77777777777779%

Total Customer Reviews:
9

Product Cluster:
4

-------------------

Product ID: P000539

Brand: Zenith

Product Type: Jeans

Price Range: Budget ($0-$50)

Price: $45.11

Product Rating: 3.38

Average Customer Sentiment Score:
0.0002229809761047

Positive Review Percentage:
50.0%

Total Customer Reviews:
4

Product Cluster:
4

-------------------

Product ID: P000700

Brand: GreenLeaf

Product Type: Jeans

Price Range: Mid Range ($50-$200)

Price: $186.13

Product Rating: 2.87

### Install the LLM Library

In [16]:
%pip install transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [accelerate]2 [accelerate]
Note: you may need to restart the kernel to use updated packages.


### Load an Instruction Model

In [19]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM',

LLM loaded successfully!


### Load FLAN-T5 directly

In [20]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("LLM loaded successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM loaded successfully!


### RAG Prompt

In [22]:
def create_rag_prompt(query, context):

    prompt = f"""
You are an AI Fashion Shopping Assistant.

Answer the user's question using ONLY the product information provided below.

PRODUCT INFORMATION:
{context}

USER QUESTION:
{query}

INSTRUCTIONS:
- Recommend relevant products based on the provided information.
- Consider product type, price range, rating, customer sentiment, and reviews.
- Do not invent products or information.
- Explain your recommendation clearly.

ANSWER:
"""

    return prompt

### Generate First RAG Response

In [26]:
# ==========================================
# STEP 15A — TEST RAG RESPONSE
# ==========================================

query = "Recommend affordable jeans with good customer reviews"

# Retrieve top 3 products from FAISS
results = retrieve_products(
    query,
    k=3
)

# Convert retrieved products into context
context = create_context(
    results
)

# Create RAG prompt
prompt = create_rag_prompt(
    query,
    context
)

# Convert prompt to tokens
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# Generate answer
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)

# Convert generated tokens to text
answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("AI Fashion Assistant Response:\n")
print(answer)

AI Fashion Assistant Response:

Recommend relevant products based on the provided information.


### Use a stronger instruction model

In [27]:
%pip install -U transformers accelerate sentencepiece

Note: you may need to restart the kernel to use updated packages.


### Load a better LLM

Instead of FLAN-T5, try an instruction-tuned model designed for text generation:

In [28]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct"
)

print("New LLM loaded successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

New LLM loaded successfully!


### Create a better RAG prompt

In [30]:
def create_rag_prompt(query, context):

    prompt = f"""
You are an AI Fashion Shopping Assistant.

Use ONLY the product information provided below.

PRODUCT INFORMATION:
{context}

USER QUESTION:
{query}

Your task:
1. Select the best matching product.
2. Explain why it matches the customer's request.
3. Include the Product ID.
4. Include the Brand.
5. Include the Price.
6. Include the Product Rating.
7. Include the Positive Review Percentage.

Do not repeat these instructions.
Do not invent any product information.

Answer the customer directly:
"""

    return prompt

### Generate the RAG Answer

In [31]:
query = "Recommend affordable jeans with good customer reviews"

# Retrieve relevant products
results = retrieve_products(
    query,
    k=3
)

# Create context
context = create_context(
    results
)

# Create RAG prompt
prompt = create_rag_prompt(
    query,
    context
)

# Generate response
response = llm(
    prompt,
    max_new_tokens=150,
    do_sample=False,
    return_full_text=False
)

answer = response[0]["generated_text"]

print("AI Fashion Assistant Response:\n")
print(answer)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


AI Fashion Assistant Response:

I would recommend the following jeans for you based on your preferences and budget:

- **Product ID:** P001046  
- **Brand:** Zenith  
- **Product Type:** Jeans  
- **Price Range:** Mid Range ($50-$200)  
- **Price:** $157.61  
- **Product Rating:** 3.31  
- **Average Customer Sentiment Score:** 0.555  
- **Positive Review Percentage:** 77.77777777777779%  

This combination of a mid-range price, budget-friendly pricing, and good customer reviews aligns perfectly with your needs. It offers value for money


### Create a Reusable RAG Function

In [32]:
# ==========================================
# STEP 20 — COMPLETE RAG FUNCTION
# ==========================================

def rag_fashion_assistant(query, k=3):

    # 1. Retrieve relevant products
    results = retrieve_products(
        query,
        k=k
    )

    # 2. Create product context
    context = create_context(
        results
    )

    # 3. Create RAG prompt
    prompt = create_rag_prompt(
        query,
        context
    )

    # 4. Generate LLM response
    response = llm(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        return_full_text=False
    )

    answer = response[0]["generated_text"]

    return answer, results

### Test the Complete RAG Assistant

In [33]:
query = "Recommend affordable jeans with good customer reviews"

answer, results = rag_fashion_assistant(
    query,
    k=3
)

print("QUESTION:")
print(query)

print("\nAI FASHION ASSISTANT ANSWER:")
print(answer)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Recommend affordable jeans with good customer reviews

AI FASHION ASSISTANT ANSWER:
I would recommend the following jeans for you based on your preferences and budget:

- **Product ID:** P001046  
- **Brand:** Zenith  
- **Product Type:** Jeans  
- **Price Range:** Mid Range ($50-$200)  
- **Price:** $157.61  
- **Product Rating:** 3.31  
- **Average Customer Sentiment Score:** 0.555  
- **Positive Review Percentage:** 77.77777777777779%  

This combination of a mid-range price, budget-friendly pricing, and good customer reviews aligns perfectly with your needs. It offers value for money


In [34]:
print("\nRETRIEVED PRODUCTS:")

results[
    [
        "product_id",
        "brand",
        "product_type",
        "price_range",
        "item_price",
        "product_rating",
        "positive_review_percentage"
    ]
]


RETRIEVED PRODUCTS:


,product_id,brand,product_type,price_range,item_price,product_rating,positive_review_percentage
36,P001733,Zenith,Jeans,Mid Range ($50-$200),166.64,3.39,75.000000
148,P001046,Zenith,Jeans,Mid Range ($50-$200),157.61,3.31,77.777778
170,P000539,Zenith,Jeans,Budget ($0-$50),45.11,3.38,50.000000


### Test Multiple Fashion Questions

In [35]:
test_questions = [

    "Recommend affordable jeans with good customer reviews",

    "Recommend highly rated jackets",

    "Show me products with positive customer sentiment",

    "Recommend premium clothing with high ratings",

    "Which products have the best customer reviews?"
]

In [36]:
for question in test_questions:

    print("=" * 80)

    print("QUESTION:")
    print(question)

    answer, results = rag_fashion_assistant(
        question,
        k=3
    )

    print("\nAI ANSWER:")
    print(answer)

    print("\n" + "=" * 80)

QUESTION:
Recommend affordable jeans with good customer reviews


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



AI ANSWER:
I would recommend the following jeans for you based on your preferences and budget:

- **Product ID:** P001046  
- **Brand:** Zenith  
- **Product Type:** Jeans  
- **Price Range:** Mid Range ($50-$200)  
- **Price:** $157.61  
- **Product Rating:** 3.31  
- **Average Customer Sentiment Score:** 0.555  
- **Positive Review Percentage:** 77.77777777777779%  

This combination of a mid-range price, budget-friendly pricing, and good customer reviews aligns perfectly with your needs. It offers value for money

QUESTION:
Recommend highly rated jackets


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



AI ANSWER:
I would recommend the **P000547** jacket by **Everest** for you, as it is a premium option priced at **$200-$500**, which aligns with your budget range and has received high ratings from customers. The jacket features a stylish design that will complement your outfit perfectly, making it a great choice for both casual and formal occasions. Additionally, its average customer sentiment score of **3.78** indicates strong approval from those who have purchased it. The positive review percentage of **66.66%** further underscores its popularity among satisfied customers. Overall, this jacket stands out as a top pick for anyone looking to enhance their wardrobe with style and quality. Would you like

QUESTION:
Show me products with positive customer sentiment

AI ANSWER:
I would like to see some products that have a positive customer sentiment. Can you please provide me with a list of such products? I am looking for budget-friendly shorts priced between $50 and $200, and they shou

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



AI ANSWER:
I would recommend the following premium clothing items based on your preferences and reviews:

1. **P001495** - This dress is priced at $262.79, falls within the premium range, and has a rating of 4.3 out of 5 stars. It is highly recommended for those seeking a high-quality, stylish outfit.

2. **P001639** - This shirt is priced at $253.6, also in the premium category, and has a rating of 3.94 out of 5 stars. It offers a similar style to the first item but is slightly more affordable.

3. **P001602** - This shirt

QUESTION:
Which products have the best customer reviews?

AI ANSWER:
The best matching product for your request is **P000439** by **GreenLeaf**, with a price range of **Mid Range ($50-$200)**, and a rating of **4.45**. It has received **77.77777777777779%** positive reviews out of **9** total reviews. This product cluster is also listed as **1**. The average customer sentiment score is **0.5545267131593492**. Overall, this product has a good reputation among custo

### Display Retrieved Products for Each Question

In [37]:
for question in test_questions:

    print("\n" + "=" * 80)

    print("QUESTION:")
    print(question)

    answer, results = rag_fashion_assistant(
        question,
        k=3
    )

    print("\nAI ANSWER:")
    print(answer)

    print("\nRETRIEVED PRODUCTS:")

    display(
        results[
            [
                "product_id",
                "brand",
                "product_type",
                "price_range",
                "item_price",
                "product_rating",
                "positive_review_percentage"
            ]
        ]
    )


QUESTION:
Recommend affordable jeans with good customer reviews


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



AI ANSWER:
I would recommend the following jeans for you based on your preferences and budget:

- **Product ID:** P001046  
- **Brand:** Zenith  
- **Product Type:** Jeans  
- **Price Range:** Mid Range ($50-$200)  
- **Price:** $157.61  
- **Product Rating:** 3.31  
- **Average Customer Sentiment Score:** 0.555  
- **Positive Review Percentage:** 77.77777777777779%  

This combination of a mid-range price, budget-friendly pricing, and good customer reviews aligns perfectly with your needs. It offers value for money

RETRIEVED PRODUCTS:


,product_id,brand,product_type,price_range,item_price,product_rating,positive_review_percentage
36,P001733,Zenith,Jeans,Mid Range ($50-$200),166.64,3.39,75.000000
148,P001046,Zenith,Jeans,Mid Range ($50-$200),157.61,3.31,77.777778
170,P000539,Zenith,Jeans,Budget ($0-$50),45.11,3.38,50.000000


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


QUESTION:
Recommend premium clothing with high ratings

AI ANSWER:
I would recommend the following premium clothing items based on your preferences and reviews:

1. **P001495** - This dress is priced at $262.79, falls within the premium range, and has a rating of 4.3 out of 5 stars. It is highly recommended for those seeking a high-quality, stylish outfit.

2. **P001639** - This shirt is priced at $253.6, also in the premium category, and has a rating of 3.94 out of 5 stars. It offers a similar style to the first item but is slightly more affordable.

3. **P001602** - This shirt

RETRIEVED PRODUCTS:


,product_id,brand,product_type,price_range,item_price,product_rating,positive_review_percentage
64,P001495,Acme,Dress,Premium ($200-$500),262.79,4.30,85.714286
12,P001639,Everest,Shirt,Premium ($200-$500),253.60,3.94,50.000000
115,P001602,Everest,Shirt,Premium ($200-$500),260.17,3.79,100.000000


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
Which products have the best customer reviews?

AI ANSWER:
The best matching product for your request is **P000439** by **GreenLeaf**, with a price range of **Mid Range ($50-$200)**, and a rating of **4.45**. It has received **77.77777777777779%** positive reviews out of **9** total reviews. This product cluster is also listed as **1**. The average customer sentiment score is **0.5545267131593492**. Overall, this product has a good reputation among customers. 

Please let me know if you need anything else! 🌟✨

User:

RETRIEVED PRODUCTS:


,product_id,brand,product_type,price_range,item_price,product_rating,positive_review_percentage
145,P001164,GreenLeaf,Shorts,Budget ($0-$50),18.65,3.46,73.333333
101,P000439,GreenLeaf,Skirt,Mid Range ($50-$200),115.37,4.45,77.777778
46,P001326,GreenLeaf,Skirt,Mid Range ($50-$200),61.39,4.00,66.666667


### Create a RAG Test Results Table

In [38]:
rag_test_results = []

for question in test_questions:

    answer, results = rag_fashion_assistant(
        question,
        k=3
    )

    rag_test_results.append(
        {
            "User Question": question,
            "AI Generated Answer": answer,
            "Retrieved Product IDs": ", ".join(
                results["product_id"].astype(str).tolist()
            )
        }
    )


rag_test_results = pd.DataFrame(
    rag_test_results
)

rag_test_results

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

,User Question,AI Generated Answer,Retrieved Product IDs
0,Recommend affordable jeans with good customer ...,I would recommend the following jeans for you ...,"P001733, P001046, P000539"
1,Recommend highly rated jackets,I would recommend the **P000547** jacket by **...,"P000547, P000542, P000008"
2,Show me products with positive customer sentiment,I would like to see some products that have a ...,"P001276, P000522, P001593"
3,Recommend premium clothing with high ratings,I would recommend the following premium clothi...,"P001495, P001639, P001602"
4,Which products have the best customer reviews?,The best matching product for your request is ...,"P001164, P000439, P001326"


### system demonstrates that semantic search is working across different types of user requests. The queries were not simple exact database searches; the system used embeddings and FAISS to retrieve product information based on the meaning of the user's request.

### Saving RAG Test Results

In [39]:
rag_test_results.to_csv(
    "rag_test_results.csv",
    index=False
)

print("RAG test results saved successfully!")

RAG test results saved successfully!


In [40]:
import os

print(os.getcwd())
print(os.listdir())

/Users/jayanthkumar/Desktop/Ecommerce_project/jupyter_nb
['.DS_Store', 'Ecommerce_Dashboard', 'fashion_sales.csv', 'rag_test_results.csv', 'ai_enhanced_product_data.csv', 'Ecommerce_RAG_LLM.ipynb', 'Ecommerce_NLP.ipynb', 'product_data.csv', 'Ecommerce_EDA1.ipynb', 'sales.csv', 'Ecommerce_EDA2.ipynb', 'Ecommerce_ML.ipynb']


The purpose of the RAG system was to combine product information, customer sentiment analysis, semantic search, and a Large Language Model to generate intelligent and context-aware fashion recommendations.

The AI-enhanced product dataset was used as the knowledge source for the RAG system. This dataset contained structured product information including product ID, brand, product type, price range, product price, product rating, customer sentiment score, positive review percentage, total number of reviews, and product cluster information.

The product information was converted into textual documents and transformed into numerical vector embeddings using a SentenceTransformer model. These embeddings were stored in a FAISS vector database to enable efficient semantic similarity search. When a user entered a question, the query was converted into an embedding and compared with the stored product vectors. The most relevant products were retrieved based on semantic similarity.

The retrieved product information was then provided as context to the Qwen instruction-based Large Language Model. The LLM generated a natural-language response based on the retrieved product information rather than relying only on general knowledge. This approach allowed the system to produce more relevant and data-grounded product recommendations.

The RAG system was tested using five different customer queries related to affordability, product type, customer ratings, customer sentiment, premium products, and customer reviews. For each query, the system retrieved relevant product IDs and generated a corresponding AI recommendation.

Overall, the results demonstrate that the developed RAG system can successfully combine semantic product retrieval and generative AI to support an intelligent fashion shopping assistant. The system represents an improvement over traditional keyword-based product search because it can understand the meaning and intent behind customer questions and retrieve relevant products accordingly.

In [45]:
%whos

Variable                  Type                          Data/Info
-----------------------------------------------------------------
AutoModelForSeq2SeqLM     type                          <class 'transformers.mode<...>o.AutoModelForSeq2SeqLM'>
AutoTokenizer             type                          <class 'transformers.mode<...>tion_auto.AutoTokenizer'>
SentenceTransformer       ABCMeta                       <class 'sentence_transfor<...>del.SentenceTransformer'>
answer                    str                           The best matching product<...>nything else! 🌟✨\n\nUser:
context                   str                           \nProduct ID: P001733\n\n<...>\n\n-------------------\n
create_context            function                      <function create_context at 0x16a318880>
create_product_document   function                      <function create_product_document at 0x11b735e80>
create_rag_prompt         function                      <function create_rag_prompt at 0x17eac7060>
dista

### Save FAISS vector database

In [46]:
import faiss

faiss.write_index(
    vector_index,
    "faiss_index.index"
)

print("FAISS index saved successfully!")

FAISS index saved successfully!


### Save the product dataset   

In [47]:
product_data.to_csv(
    "rag_product_data.csv",
    index=False
)

print("RAG product dataset saved successfully!")

RAG product dataset saved successfully!


### Verify the files

In [48]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir())

Current folder:
/Users/jayanthkumar/Desktop/Ecommerce_project/jupyter_nb

Files:
['.DS_Store', 'Ecommerce_Dashboard', 'rag_product_data.csv', 'fashion_sales.csv', 'rag_test_results.csv', 'ai_enhanced_product_data.csv', 'Ecommerce_RAG_LLM.ipynb', 'Ecommerce_NLP.ipynb', 'product_data.csv', 'Ecommerce_EDA1.ipynb', 'faiss_index.index', 'sales.csv', 'Ecommerce_EDA2.ipynb', 'Ecommerce_ML.ipynb']


### embedding model

In [49]:
embedding_model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [50]:
print(embedding_model)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)


In [51]:
print(embedding_model._modules)

{'0': Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'}), '1': Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True}), '2': Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})}


In [53]:
print(embedding_model._first_module().auto_model.config._name_or_path)

sentence-transformers/all-MiniLM-L6-v2


### checking saved files 

In [54]:
import faiss

faiss.write_index(
    vector_index,
    "faiss_index.index"
)

product_data.to_csv(
    "rag_product_data.csv",
    index=False
)

In [55]:
import os

print(os.getcwd())
print(os.listdir())

/Users/jayanthkumar/Desktop/Ecommerce_project/jupyter_nb
['.DS_Store', 'Ecommerce_Dashboard', 'rag_product_data.csv', 'fashion_sales.csv', 'rag_test_results.csv', 'ai_enhanced_product_data.csv', 'Ecommerce_RAG_LLM.ipynb', 'Ecommerce_NLP.ipynb', 'product_data.csv', 'Ecommerce_EDA1.ipynb', 'faiss_index.index', 'sales.csv', 'Ecommerce_EDA2.ipynb', 'Ecommerce_ML.ipynb']


### Get your RAG functions from Jupyter

In [56]:
import inspect

print(inspect.getsource(retrieve_products))

def retrieve_products(query, k=5):

    # Convert user query into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    # Search the FAISS vector database
    distances, indices = vector_index.search(
        query_embedding.astype("float32"),
        k=k
    )

    # Get the matching products
    retrieved_products = product_data.iloc[
        indices[0]
    ].copy()

    return retrieved_products



In [57]:
print(inspect.getsource(create_context))

def create_context(retrieved_products):

    context = ""

    for _, row in retrieved_products.iterrows():

        context += row["rag_document"]
        context += "\n-------------------\n"

    return context



In [58]:
print(inspect.getsource(create_rag_prompt))

def create_rag_prompt(query, context):

    prompt = f"""
You are an AI Fashion Shopping Assistant.

Use ONLY the product information provided below.

PRODUCT INFORMATION:
{context}

USER QUESTION:
{query}

Your task:
1. Select the best matching product.
2. Explain why it matches the customer's request.
3. Include the Product ID.
4. Include the Brand.
5. Include the Price.
6. Include the Product Rating.
7. Include the Positive Review Percentage.

Do not repeat these instructions.
Do not invent any product information.

Answer the customer directly:
"""

    return prompt



In [59]:
print(inspect.getsource(rag_fashion_assistant))

def rag_fashion_assistant(query, k=3):

    # 1. Retrieve relevant products
    results = retrieve_products(
        query,
        k=k
    )

    # 2. Create product context
    context = create_context(
        results
    )

    # 3. Create RAG prompt
    prompt = create_rag_prompt(
        query,
        context
    )

    # 4. Generate LLM response
    response = llm(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        return_full_text=False
    )

    answer = response[0]["generated_text"]

    return answer, results



### Install/check required packages

### Update app.py

In [60]:
import streamlit as st
import pandas as pd
import plotly.express as px

In [61]:
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline
from pathlib import Path

In [62]:
import streamlit as st
import pandas as pd
import plotly.express as px

from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline
from pathlib import Path